# Full Model Sweep: CNN / ANN / LSTM

Self-contained Colab notebook. No repo clone required.

**What it does:**
1. Loads cached windows from Google Drive
2. Trains CNN, ANN, and LSTM on watch / fusion / phone channels
3. Saves checkpoints + metrics back to Drive
4. Generates a comparison table

**Setup:**
- Upload `artifacts/windows/` to `MyDrive/cs286-project/artifacts/windows/`
- Runtime → Change runtime type → T4 GPU
- Run all cells

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# Configuration
# ═══════════════════════════════════════════════════════════════════════════════
WINDOWS_ROOT = '/content/drive/MyDrive/cs286-project/artifacts/windows'
OUTPUT_ROOT  = '/content/drive/MyDrive/cs286-project-runs/full_sweep'

SEED = 42
BATCH_SIZE = 256
LR = 1e-3
WEIGHT_DECAY = 1e-4
MAX_EPOCHS = 40
PATIENCE = 7
GRAD_CLIP = 1.0
DROPOUT = 0.2
LABEL_FRACTION = 1.0
AUGMENT = True

MODEL_TYPES = ['cnn', 'ann', 'lstm']
CHANNEL_MODES = ['watch', 'fusion', 'phone']

print('Output root:', OUTPUT_ROOT)

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# Imports & helpers
# ═══════════════════════════════════════════════════════════════════════════════
import json, csv, math, random, pickle, time
from collections import defaultdict
from pathlib import Path
from typing import Dict, List, Tuple

import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

def seed_everything(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def detect_device():
    if torch.cuda.is_available():
        return torch.device('cuda')
    if getattr(torch.backends, 'mps', None) is not None and torch.backends.mps.is_available():
        return torch.device('mps')
    return torch.device('cpu')

def metric_token(value: float) -> str:
    return f"{value:.4f}".replace('.', 'p')

device = detect_device()
print('Device:', device)

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# Data loading
# ═══════════════════════════════════════════════════════════════════════════════
window_root = Path(WINDOWS_ROOT)
manifest = json.loads((window_root / 'manifest.json').read_text())
label_to_index = manifest['label_to_index']
index_to_label = {i: l for l, i in label_to_index.items()}
num_classes = len(label_to_index)
print('Classes:', num_classes, list(label_to_index.keys()))

def read_metadata_rows(split: str) -> List[dict]:
    path = window_root / split / 'metadata.csv'
    with path.open('r', encoding='utf-8', newline='') as f:
        reader = csv.DictReader(f)
        return [r for r in reader]

def read_window_payloads(split: str) -> Dict[int, list]:
    payloads = {}
    for chunk_path in sorted((window_root / split).glob('data_chunk_*.pkl')):
        with chunk_path.open('rb') as f:
            records = pickle.load(f)
        for r in records:
            payloads[int(r['window_id'])] = r['x_fusion']
    return payloads

def select_labeled_subset(rows: List[dict], fraction: float, seed: int) -> List[dict]:
    if fraction >= 1.0:
        return rows
    grouped = defaultdict(list)
    for r in rows:
        grouped[(int(r['subject_id']), int(r['label_idx']))].append(r)
    selected = []
    for gi, key in enumerate(sorted(grouped)):
        grp = grouped[key]
        rng = random.Random(seed + gi * 9973)
        quota = max(1, math.ceil(len(grp) * fraction))
        selected.extend(rng.sample(grp, min(quota, len(grp))))
    selected.sort(key=lambda r: int(r['window_id']))
    return selected

CHANNEL_SLICES = {'fusion': slice(0, 12), 'phone': slice(0, 6), 'watch': slice(6, 12)}

def build_arrays(rows, payloads, mode):
    sl = CHANNEL_SLICES[mode]
    x = np.stack([payloads[int(r['window_id'])][sl] for r in rows]).astype(np.float32)
    y = np.asarray([int(r['label_idx']) for r in rows], dtype=np.int64)
    s = np.asarray([int(r['subject_id']) for r in rows], dtype=np.int64)
    return x, y, s

# Load everything once
train_rows_all = read_metadata_rows('train')
val_rows   = read_metadata_rows('val')
test_rows  = read_metadata_rows('test')
payloads = {s: read_window_payloads(s) for s in ('train', 'val', 'test')}
train_rows = select_labeled_subset(train_rows_all, LABEL_FRACTION, SEED)

print(f'Train: {len(train_rows)}  Val: {len(val_rows)}  Test: {len(test_rows)}')

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# Model definitions (inline, self-contained)
# ═══════════════════════════════════════════════════════════════════════════════
class TimeSeriesEncoder(nn.Module):
    def __init__(self, in_channels: int):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv1d(in_channels, 64, 5, padding=2),
            nn.BatchNorm1d(64), nn.ReLU(), nn.MaxPool1d(2),
            nn.Conv1d(64, 128, 5, padding=2),
            nn.BatchNorm1d(128), nn.ReLU(), nn.MaxPool1d(2),
            nn.Conv1d(128, 256, 3, padding=1),
            nn.BatchNorm1d(256), nn.ReLU(),
            nn.AdaptiveAvgPool1d(1),
        )
    def forward(self, x):
        return self.features(x).squeeze(-1)

class SupervisedHARModel(nn.Module):
    def __init__(self, in_channels, num_classes, dropout=0.2):
        super().__init__()
        self.encoder = TimeSeriesEncoder(in_channels)
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(256, num_classes)
        )
    def forward(self, x):
        return self.classifier(self.encoder(x))

class SupervisedFusionModel(nn.Module):
    def __init__(self, num_classes, dropout=0.2):
        super().__init__()
        self.phone_encoder = TimeSeriesEncoder(in_channels=6)
        self.watch_encoder = TimeSeriesEncoder(in_channels=6)
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(512, num_classes)
        )
    def forward(self, x):
        h_phone = self.phone_encoder(x[:, :6, :])
        h_watch = self.watch_encoder(x[:, 6:, :])
        return self.classifier(torch.cat([h_phone, h_watch], dim=1))

class SupervisedANNModel(nn.Module):
    def __init__(self, in_channels, num_classes, dropout=0.2):
        super().__init__()
        self.flatten = nn.Flatten(start_dim=1)
        self.classifier = nn.Sequential(
            nn.Linear(in_channels * 60, 512), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(512, 256), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(256, num_classes)
        )
    def forward(self, x):
        return self.classifier(self.flatten(x))

class SupervisedLSTMModel(nn.Module):
    def __init__(self, in_channels, num_classes, dropout=0.2):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=in_channels, hidden_size=128,
            num_layers=2, batch_first=True, dropout=dropout
        )
        self.classifier = nn.Linear(128, num_classes)
    def forward(self, x):
        x = x.permute(0, 2, 1)           # (B, C, T) -> (B, T, C)
        out, _ = self.lstm(x)            # (B, T, H)
        return self.classifier(out.mean(dim=1))

def build_model(model_type, in_channels, num_classes, dropout=0.2):
    if model_type == 'cnn':
        if in_channels == 12:
            return SupervisedFusionModel(num_classes, dropout)
        return SupervisedHARModel(in_channels, num_classes, dropout)
    if model_type == 'ann':
        return SupervisedANNModel(in_channels, num_classes, dropout)
    if model_type == 'lstm':
        return SupervisedLSTMModel(in_channels, num_classes, dropout)
    raise ValueError(model_type)

print('Model definitions ready.')

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# Training / evaluation utilities
# ═══════════════════════════════════════════════════════════════════════════════
class TimeSeriesAugmentationConfig:
    def __init__(self, jitter_std=0.01, scale_min=0.9, scale_max=1.1,
                 mask_ratio=0.1, mask_length=6, enabled=True):
        self.jitter_std = jitter_std
        self.scale_min = scale_min
        self.scale_max = scale_max
        self.mask_ratio = mask_ratio
        self.mask_length = mask_length
        self.enabled = enabled

def apply_augmentations(x, config):
    if not config.enabled:
        return x
    aug = x.clone()
    if config.jitter_std > 0:
        aug = aug + (config.jitter_std * torch.randn_like(aug))
    scales = torch.empty((aug.shape[0], 1, 1), device=aug.device, dtype=aug.dtype)
    scales.uniform_(config.scale_min, config.scale_max)
    aug = aug * scales
    steps = aug.shape[-1]
    max_masks = int(round(steps * config.mask_ratio / max(1, config.mask_length)))
    num_masks = max(1, max_masks)
    max_start = max(1, steps - config.mask_length + 1)
    for b in range(aug.shape[0]):
        for _ in range(num_masks):
            start = torch.randint(0, max_start, (1,), device=aug.device).item()
            end = min(steps, start + config.mask_length)
            aug[b, :, start:end] = 0.0
    return aug

def to_loader(x, y, batch_size, shuffle):
    return DataLoader(TensorDataset(torch.from_numpy(x), torch.from_numpy(y)),
                      batch_size=batch_size, shuffle=shuffle, num_workers=0)

def compute_metrics(y_true, y_pred, num_classes, subjects):
    matrix = np.zeros((num_classes, num_classes), dtype=np.int64)
    for t, p in zip(y_true, y_pred):
        matrix[t, p] += 1
    acc = matrix.diagonal().sum() / max(1, matrix.sum())
    per_class_f1 = {}
    f1s = []
    for c in range(num_classes):
        tp = matrix[c, c]
        fp = matrix[:, c].sum() - tp
        fn = matrix[c, :].sum() - tp
        prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        rec  = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        f1   = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0.0
        per_class_f1[index_to_label[c]] = f1
        f1s.append(f1)
    macro_f1 = sum(f1s) / len(f1s)

    subj_acc = {}
    counts = defaultdict(lambda: [0, 0])
    for sid, t, p in zip(subjects, y_true, y_pred):
        counts[int(sid)][0] += int(t == p)
        counts[int(sid)][1] += 1
    for sid, (c, t) in sorted(counts.items()):
        subj_acc[sid] = c / t
    return {'accuracy': float(acc), 'macro_f1': float(macro_f1),
            'per_class_f1': per_class_f1, 'confusion_matrix': matrix,
            'per_subject_accuracy': subj_acc}

def evaluate(model, loader, loss_fn, subjects):
    model.eval()
    total_loss = 0.0
    all_true, all_pred = [], []
    with torch.no_grad():
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            logits = model(xb)
            total_loss += float(loss_fn(logits, yb).item()) * len(xb)
            all_true.extend(yb.cpu().tolist())
            all_pred.extend(logits.argmax(1).cpu().tolist())
    m = compute_metrics(all_true, all_pred, num_classes, subjects)
    m['loss'] = total_loss / len(all_true)
    return m

def train_epoch(model, loader, optimizer, loss_fn, aug_config=None):
    model.train()
    total_loss = 0.0
    total = 0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        if aug_config is not None:
            xb = apply_augmentations(xb, aug_config)
        optimizer.zero_grad()
        loss = loss_fn(model(xb), yb)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        optimizer.step()
        total_loss += float(loss.item()) * len(xb)
        total += len(xb)
    return total_loss / max(1, total)

print('Training utilities ready.')

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# Sweep execution
# ═══════════════════════════════════════════════════════════════════════════════
output_root = Path(OUTPUT_ROOT)
output_root.mkdir(parents=True, exist_ok=True)

results = []

for model_type in MODEL_TYPES:
    for channel_mode in CHANNEL_MODES:
        run_name = f"{channel_mode}_full_{model_type}" if model_type != 'cnn' else f"{channel_mode}_full"
        run_dir = output_root / f"supervised-baseline-{channel_mode}-{model_type}"
        run_dir.mkdir(parents=True, exist_ok=True)
        
        print("\n" + "=" * 60)
        print(f"  STARTING: {model_type.upper()} | {channel_mode.upper()}")
        print("=" * 60)

        x_tr, y_tr, s_tr = build_arrays(train_rows, payloads['train'], channel_mode)
        x_va, y_va, s_va = build_arrays(val_rows,   payloads['val'],   channel_mode)
        x_te, y_te, s_te = build_arrays(test_rows,  payloads['test'],  channel_mode)

        train_loader = to_loader(x_tr, y_tr, BATCH_SIZE, shuffle=True)
        val_loader   = to_loader(x_va, y_va, BATCH_SIZE, shuffle=False)
        test_loader  = to_loader(x_te, y_te, BATCH_SIZE, shuffle=False)

        seed_everything(SEED)
        model = build_model(model_type, x_tr.shape[1], num_classes, DROPOUT).to(device)

        class_counts = np.bincount(y_tr, minlength=num_classes)
        class_weights = torch.tensor(1.0 / (class_counts.astype(np.float32) + 1.0), dtype=torch.float32)
        class_weights = class_weights / class_weights.sum() * num_classes
        class_weights = class_weights.to(device)
        loss_fn = nn.CrossEntropyLoss(weight=class_weights)
        optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=3)
        aug_config = TimeSeriesAugmentationConfig(enabled=AUGMENT)

        best_f1 = -1.0
        best_state = None
        best_epoch = -1
        epochs_no_improve = 0
        history = []

        for epoch in range(1, MAX_EPOCHS + 1):
            t0 = time.time()
            tr_loss = train_epoch(model, train_loader, optimizer, loss_fn, aug_config)
            val_m = evaluate(model, val_loader, loss_fn, s_va)
            history.append({
                'epoch': epoch, 'train_loss': tr_loss,
                'val_loss': val_m['loss'], 'val_accuracy': val_m['accuracy'],
                'val_macro_f1': val_m['macro_f1']
            })
            print(f"[{run_name}] e={epoch:02d} tr_loss={tr_loss:.4f} "
                  f"val_loss={val_m['loss']:.4f} val_acc={val_m['accuracy']:.4f} "
                  f"val_f1={val_m['macro_f1']:.4f} ({time.time()-t0:.1f}s)")

            if val_m['macro_f1'] > best_f1:
                best_f1 = val_m['macro_f1']
                best_epoch = epoch
                best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
                epochs_no_improve = 0
            else:
                epochs_no_improve += 1
                if epochs_no_improve >= PATIENCE:
                    print(f"[{run_name}] early stopping at epoch {epoch}")
                    break
            scheduler.step(val_m['macro_f1'])

        model.load_state_dict(best_state)
        train_m = evaluate(model, train_loader, loss_fn, s_tr)
        val_m   = evaluate(model, val_loader,   loss_fn, s_va)
        test_m  = evaluate(model, test_loader,  loss_fn, s_te)

        # Save checkpoint
        stem = (f"supervised_{channel_mode}_{model_type}_baseline_{run_name}_"
                f"acc{metric_token(test_m['accuracy'])}_macrof1{metric_token(test_m['macro_f1'])}")
        ckpt_path = run_dir / f"{stem}_checkpoint.pt"
        torch.save({
            'model_state_dict': model.state_dict(),
            'best_epoch': best_epoch,
            'label_to_index': label_to_index,
            'channel_mode': channel_mode,
            'model_type': model_type,
            'label_fraction': LABEL_FRACTION,
            'experiment_name': run_name,
        }, ckpt_path)

        # Save metrics
        metrics_payload = {
            'experiment_name': run_name,
            'channel_mode': channel_mode,
            'model_type': model_type,
            'label_fraction': LABEL_FRACTION,
            'best_epoch': best_epoch,
            'config': {'batch_size': BATCH_SIZE, 'learning_rate': LR,
                       'weight_decay': WEIGHT_DECAY, 'dropout': DROPOUT,
                       'epochs_requested': MAX_EPOCHS, 'patience': PATIENCE,
                       'lr_scheduler': 'ReduceLROnPlateau', 'lr_scheduler_patience': 3, 'lr_scheduler_factor': 0.5,
                       'augment': AUGMENT, 'class_weights': class_weights.cpu().numpy().round(6).tolist(),
                       'channel_mode': channel_mode, 'model_type': model_type,
                       'input_channels': int(x_tr.shape[1]),
                       'label_fraction': LABEL_FRACTION, 'seed': SEED},
            'train': {'loss': train_m['loss'], 'accuracy': train_m['accuracy'], 'macro_f1': train_m['macro_f1']},
            'val':   {'loss': val_m['loss'],   'accuracy': val_m['accuracy'],   'macro_f1': val_m['macro_f1']},
            'test':  {'loss': test_m['loss'],  'accuracy': test_m['accuracy'],  'macro_f1': test_m['macro_f1'],
                      'per_class_f1': test_m['per_class_f1'],
                      'per_subject_accuracy': test_m['per_subject_accuracy'],
                      'confusion_matrix': test_m['confusion_matrix'].tolist()},
            'history': history,
            'artifacts': {'checkpoint_path': str(ckpt_path)}
        }
        metrics_path = run_dir / f"{stem}_metrics.json"
        metrics_path.write_text(json.dumps(metrics_payload, indent=2), encoding='utf-8')

        results.append({
            'model_type': model_type, 'channel_mode': channel_mode,
            'run_name': run_name, 'best_epoch': best_epoch,
            'test_accuracy': test_m['accuracy'], 'test_macro_f1': test_m['macro_f1'],
            'metrics_path': str(metrics_path), 'checkpoint_path': str(ckpt_path)
        })

        print(f"[{run_name}] DONE — test_acc={test_m['accuracy']:.4f} test_f1={test_m['macro_f1']:.4f}")

# Save summary CSV
summary_path = output_root / 'sweep_summary.csv'
with summary_path.open('w', encoding='utf-8', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=results[0].keys())
    writer.writeheader()
    writer.writerows(results)
print(f'\nSummary written to: {summary_path}')

# Section A: UCI HAR Download & Preprocessing

Download and prepare the UCI HAR Dataset for transfer learning pretraining.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# UCI HAR Transfer Learning Configuration
# ═══════════════════════════════════════════════════════════════════════════════
UCI_HAR_ROOT = '/content/uci_har'
PRETRAIN_OUTPUT = '/content/drive/MyDrive/cs286-project-runs/pretrained'

UCI_HAR_URL = 'https://archive.ics.uci.edu/ml/machine-learning-databases/00240/UCI%20HAR%20Dataset.zip'

UCI_NUM_CLASSES = 6
UCI_CHANNELS = 6
UCI_TIMESTEPS = 128

PRETRAIN_EPOCHS = 40
PRETRAIN_LR = 1e-3
PRETRAIN_BATCH_SIZE = 256

print('UCI HAR config ready.')

In [ ]:
import os, zipfile
from pathlib import Path

uci_root = Path(UCI_HAR_ROOT)
uci_root.mkdir(parents=True, exist_ok=True)
zip_path = uci_root / 'UCI_HAR_Dataset.zip'

if not zip_path.exists():
    print('Downloading UCI HAR...')
    !wget -q "{UCI_HAR_URL}" -O "{zip_path}"
    print('Download complete.')
else:
    print('UCI HAR zip already exists.')

extracted_dir = uci_root / 'UCI HAR Dataset'
if not extracted_dir.exists():
    print('Extracting...')
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall(uci_root)
    print('Extraction complete.')
else:
    print('Already extracted.')

print(f'Data directory: {extracted_dir}')

In [ ]:
def load_uci_har_signals(split='train'):
    signal_dir = extracted_dir / 'Inertial Signals'
    channels = []
    for prefix in ['body_acc', 'body_gyro']:
        for axis in ['x', 'y', 'z']:
            file_path = signal_dir / f'{prefix}_{axis}_{split}.txt'
            data = np.loadtxt(file_path)
            channels.append(data)
    x = np.stack(channels, axis=1).astype(np.float32)
    label_path = extracted_dir / split / f'y_{split}.txt'
    y = np.loadtxt(label_path).astype(np.int64) - 1
    return x, y

uci_x_train, uci_y_train = load_uci_har_signals('train')
uci_x_test, uci_y_test = load_uci_har_signals('test')
print(f'UCI HAR train: {uci_x_train.shape}, labels: {uci_y_train.shape}')
print(f'UCI HAR test:  {uci_x_test.shape}, labels: {uci_y_test.shape}')

# Section B: UCI HAR Pretraining

Pretrain CNN and LSTM encoders on the UCI HAR dataset.

In [ ]:
def build_uci_model(model_type, num_classes=UCI_NUM_CLASSES, dropout=DROPOUT):
    return build_model(model_type, in_channels=UCI_CHANNELS, num_classes=num_classes, dropout=dropout)
print('UCI HAR model builder ready.')

In [ ]:
def pretrain_on_uci(model_type, epochs=PRETRAIN_EPOCHS, lr=PRETRAIN_LR):
    train_dataset = TensorDataset(torch.from_numpy(uci_x_train), torch.from_numpy(uci_y_train))
    test_dataset = TensorDataset(torch.from_numpy(uci_x_test), torch.from_numpy(uci_y_test))
    train_loader = DataLoader(train_dataset, batch_size=PRETRAIN_BATCH_SIZE, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=PRETRAIN_BATCH_SIZE, shuffle=False)
    
    model = build_uci_model(model_type).to(device)
    loss_fn = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=3)
    
    best_acc = 0.0
    best_state = None
    
    for epoch in range(1, epochs + 1):
        model.train()
        train_loss = 0.0
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            loss = loss_fn(model(xb), yb)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            optimizer.step()
            train_loss += float(loss.item()) * len(xb)
        train_loss /= len(train_loader.dataset)
        
        model.eval()
        correct = total = 0
        with torch.no_grad():
            for xb, yb in test_loader:
                xb, yb = xb.to(device), yb.to(device)
                pred = model(xb).argmax(1)
                correct += int((pred == yb).sum().item())
                total += len(yb)
        
        acc = correct / total
        scheduler.step(acc)
        
        if acc > best_acc:
            best_acc = acc
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        
        if epoch % 5 == 0 or epoch == 1:
            print(f'[UCI {model_type.upper()}] epoch={epoch:02d} train_loss={train_loss:.4f} test_acc={acc:.4f}')
    
    print(f'[UCI {model_type.upper()}] Best test accuracy: {best_acc:.4f}')
    return best_state, best_acc

print('Pretraining function ready.')

In [ ]:
pretrain_output = Path(PRETRAIN_OUTPUT)
pretrain_output.mkdir(parents=True, exist_ok=True)

print('\n' + '='*60)
print('PRETRAINING CNN ON UCI HAR')
print('='*60)
cnn_state, cnn_acc = pretrain_on_uci('cnn')

cnn_encoder_path = pretrain_output / 'uci_har_cnn_encoder.pt'
torch.save({
    'encoder_state_dict': {k: v for k, v in cnn_state.items() if k.startswith('encoder.')},
    'full_model_state_dict': cnn_state,
    'model_type': 'cnn',
    'source_dataset': 'uci_har',
    'source_classes': UCI_NUM_CLASSES,
    'source_channels': UCI_CHANNELS,
    'source_timesteps': UCI_TIMESTEPS,
    'test_accuracy': cnn_acc,
}, cnn_encoder_path)
print(f'Saved CNN encoder to: {cnn_encoder_path}')

print('\n' + '='*60)
print('PRETRAINING LSTM ON UCI HAR')
print('='*60)
lstm_state, lstm_acc = pretrain_on_uci('lstm')

lstm_encoder_path = pretrain_output / 'uci_har_lstm_encoder.pt'
torch.save({
    'encoder_state_dict': {k: v for k, v in lstm_state.items() if k.startswith('lstm.')},
    'full_model_state_dict': lstm_state,
    'model_type': 'lstm',
    'source_dataset': 'uci_har',
    'source_classes': UCI_NUM_CLASSES,
    'source_channels': UCI_CHANNELS,
    'source_timesteps': UCI_TIMESTEPS,
    'test_accuracy': lstm_acc,
}, lstm_encoder_path)
print(f'Saved LSTM encoder to: {lstm_encoder_path}')

# Section C: Transfer Learning Utilities

Helper functions for loading pretrained encoders and applying freeze strategies.

In [ ]:
def load_pretrained_encoder(model, encoder_path, model_type):
    checkpoint = torch.load(encoder_path, map_location='cpu')
    encoder_state = checkpoint['encoder_state_dict']
    model.load_state_dict(encoder_state, strict=False)
    print(f'Loaded {len(encoder_state)} encoder parameters from {encoder_path}')
    return model

def apply_freeze_strategy(model, strategy, model_type):
    if strategy == 'none':
        for p in model.parameters():
            p.requires_grad = True
        return 'No freezing'
    elif strategy == 'all':
        if model_type == 'cnn':
            for p in model.encoder.parameters():
                p.requires_grad = False
        elif model_type == 'lstm':
            for p in model.lstm.parameters():
                p.requires_grad = False
        return 'Frozen entire encoder'
    elif strategy == 'first_two':
        if model_type == 'cnn':
            for i, layer in enumerate(model.encoder.features):
                for p in layer.parameters():
                    p.requires_grad = i >= 8
        elif model_type == 'lstm':
            for name, p in model.lstm.named_parameters():
                p.requires_grad = 'l1' in name
        return 'Frozen first two blocks/layer'
    elif strategy == 'progressive':
        if model_type == 'cnn':
            for p in model.encoder.parameters():
                p.requires_grad = False
        elif model_type == 'lstm':
            for p in model.lstm.parameters():
                p.requires_grad = False
        return 'Progressive freeze'
    else:
        raise ValueError(strategy)

def unfreeze_all(model, model_type):
    if model_type == 'cnn':
        for p in model.encoder.parameters():
            p.requires_grad = True
    elif model_type == 'lstm':
        for p in model.lstm.parameters():
            p.requires_grad = True

def count_trainable(model):
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    return trainable, total

print('Transfer utilities ready.')

# Section D: Transfer Learning Sweep

Fine-tune pretrained encoders on the target dataset with various freeze strategies.

In [ ]:
TRANSFER_OUTPUT = '/content/drive/MyDrive/cs286-project-runs/transfer_sweep'
TRANSFER_CHANNEL = 'watch'
TRANSFER_IN_CHANNELS = 6
FINETUNE_LR = 1e-4
FINETUNE_EPOCHS = 40
FINETUNE_PATIENCE = 7

TRANSFER_CONFIGS = [
    {'model_type': 'cnn', 'freeze': 'none', 'name': 'cnn_scratch', 'pretrained': None},
    {'model_type': 'lstm', 'freeze': 'none', 'name': 'lstm_scratch', 'pretrained': None},
    {'model_type': 'cnn', 'freeze': 'all', 'name': 'cnn_frozen', 'pretrained': str(cnn_encoder_path)},
    {'model_type': 'lstm', 'freeze': 'all', 'name': 'lstm_frozen', 'pretrained': str(lstm_encoder_path)},
    {'model_type': 'cnn', 'freeze': 'first_two', 'name': 'cnn_partial', 'pretrained': str(cnn_encoder_path)},
    {'model_type': 'lstm', 'freeze': 'first_two', 'name': 'lstm_partial', 'pretrained': str(lstm_encoder_path)},
    {'model_type': 'cnn', 'freeze': 'progressive', 'name': 'cnn_progressive', 'pretrained': str(cnn_encoder_path), 'unfreeze_at': 10},
    {'model_type': 'lstm', 'freeze': 'progressive', 'name': 'lstm_progressive', 'pretrained': str(lstm_encoder_path), 'unfreeze_at': 10},
]

In [ ]:
def train_with_transfer(config, x_tr, y_tr, x_va, y_va, x_te, y_te, s_va, s_te):
    model_type = config['model_type']
    freeze_strategy = config['freeze']
    pretrained_path = config.get('pretrained')
    unfreeze_at = config.get('unfreeze_at')
    run_name = config['name']
    
    seed_everything(SEED)
    model = build_model(model_type, TRANSFER_IN_CHANNELS, num_classes, DROPOUT).to(device)
    
    if pretrained_path is not None:
        model = load_pretrained_encoder(model, pretrained_path, model_type)
    
    freeze_desc = apply_freeze_strategy(model, freeze_strategy, model_type)
    trainable, total = count_trainable(model)
    print(f"[{run_name}] {freeze_desc} — {trainable:,}/{total:,} trainable")
    
    train_loader = to_loader(x_tr, y_tr, BATCH_SIZE, shuffle=True)
    val_loader = to_loader(x_va, y_va, BATCH_SIZE, shuffle=False)
    test_loader = to_loader(x_te, y_te, BATCH_SIZE, shuffle=False)
    
    class_counts = np.bincount(y_tr, minlength=num_classes)
    class_weights = torch.tensor(1.0 / (class_counts.astype(np.float32) + 1.0), dtype=torch.float32)
    class_weights = class_weights / class_weights.sum() * num_classes
    class_weights = class_weights.to(device)
    loss_fn = nn.CrossEntropyLoss(weight=class_weights)
    
    trainable_params = filter(lambda p: p.requires_grad, model.parameters())
    lr = FINETUNE_LR if pretrained_path is not None else LR
    optimizer = torch.optim.AdamW(trainable_params, lr=lr, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=3)
    aug_config = TimeSeriesAugmentationConfig(enabled=AUGMENT)
    
    best_f1 = -1.0
    best_state = None
    best_epoch = -1
    epochs_no_improve = 0
    history = []
    
    for epoch in range(1, FINETUNE_EPOCHS + 1):
        if freeze_strategy == 'progressive' and unfreeze_at is not None and epoch == unfreeze_at:
            unfreeze_all(model, model_type)
            trainable_params = filter(lambda p: p.requires_grad, model.parameters())
            optimizer = torch.optim.AdamW(trainable_params, lr=lr * 0.1, weight_decay=WEIGHT_DECAY)
            print(f"[{run_name}] Unfreeze at epoch {epoch}, LR={lr*0.1:.1e}")
        
        tr_loss = train_epoch(model, train_loader, optimizer, loss_fn, aug_config)
        val_m = evaluate(model, val_loader, loss_fn, s_va)
        history.append({'epoch': epoch, 'train_loss': tr_loss, 'val_loss': val_m['loss'], 
                        'val_accuracy': val_m['accuracy'], 'val_macro_f1': val_m['macro_f1']})
        print(f"[{run_name}] e={epoch:02d} tr_loss={tr_loss:.4f} val_acc={val_m['accuracy']:.4f} val_f1={val_m['macro_f1']:.4f}")
        
        if val_m['macro_f1'] > best_f1:
            best_f1 = val_m['macro_f1']
            best_epoch = epoch
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= FINETUNE_PATIENCE:
                print(f"[{run_name}] early stopping at epoch {epoch}")
                break
        scheduler.step(val_m['macro_f1'])
    
    model.load_state_dict(best_state)
    test_m = evaluate(model, test_loader, loss_fn, s_te)
    print(f"[{run_name}] DONE — test_acc={test_m['accuracy']:.4f} test_f1={test_m['macro_f1']:.4f}")
    
    return {'model_type': model_type, 'freeze_strategy': freeze_strategy, 'run_name': run_name,
            'pretrained': pretrained_path is not None, 'best_epoch': best_epoch,
            'test_accuracy': test_m['accuracy'], 'test_macro_f1': test_m['macro_f1'],
            'trainable_params': trainable, 'total_params': total, 'history': history, 'test_metrics': test_m}

print('Transfer training function ready.')

In [ ]:
transfer_output = Path(TRANSFER_OUTPUT)
transfer_output.mkdir(parents=True, exist_ok=True)

x_tr_w, y_tr_w, s_tr_w = build_arrays(train_rows, payloads['train'], 'watch')
x_va_w, y_va_w, s_va_w = build_arrays(val_rows, payloads['val'], 'watch')
x_te_w, y_te_w, s_te_w = build_arrays(test_rows, payloads['test'], 'watch')

transfer_results = []
for config in TRANSFER_CONFIGS:
    print('\n' + '='*70)
    print(f"  STARTING: {config['name']}")
    print('='*70)
    result = train_with_transfer(config, x_tr_w, y_tr_w, x_va_w, y_va_w, x_te_w, y_te_w, s_va_w, s_te_w)
    transfer_results.append(result)
    result_path = transfer_output / f"{config['name']}_result.json"
    result_serializable = {k: v for k, v in result.items() if k != 'test_metrics'}
    result_serializable['test_metrics'] = {'accuracy': result['test_metrics']['accuracy'], 'macro_f1': result['test_metrics']['macro_f1'], 'per_class_f1': result['test_metrics']['per_class_f1']}
    result_path.write_text(json.dumps(result_serializable, indent=2), encoding='utf-8')

summary_path = transfer_output / 'transfer_summary.json'
summary_path.write_text(json.dumps({'transfer_results': transfer_results, 'configs': TRANSFER_CONFIGS}, indent=2, default=str), encoding='utf-8')
print(f'\nTransfer sweep complete. Summary: {summary_path}')

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# Print pretty comparison table
# ═══════════════════════════════════════════════════════════════════════════════
print('\n' + '='*70)
print('  FINAL COMPARISON TABLE')
print('='*70)
print(f"{'Model':<8} {'Channel':<10} {'Test Acc':>10} {'Test F1':>10} {'Epoch':>6}")
print('-'*70)
for r in results:
    print(f"{r['model_type']:<8} {r['channel_mode']:<10} {r['test_accuracy']:>10.4f} {r['test_macro_f1']:>10.4f} {r['best_epoch']:>6}")
print('='*70)

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# Figure 1: Grouped bar chart — all model/channel combinations
# ═══════════════════════════════════════════════════════════════════════════════
import seaborn as sns
sns.set_style('whitegrid')

fig, ax = plt.subplots(figsize=(12, 5))
x = np.arange(len(results))
width = 0.35
acc_vals = [r['test_accuracy'] for r in results]
f1_vals  = [r['test_macro_f1'] for r in results]
labels = [f"{r['model_type'].upper()}\n{r['channel_mode']}" for r in results]

bars1 = ax.bar(x - width/2, acc_vals, width, label='Test Accuracy', color='steelblue')
bars2 = ax.bar(x + width/2, f1_vals,  width, label='Test Macro F1', color='coral')
ax.set_ylabel('Score')
ax.set_title('Test Accuracy vs Macro F1 — Full Model Sweep')
ax.set_xticks(x)
ax.set_xticklabels(labels, fontsize=8)
ax.legend()
ax.set_ylim(0, 1.0)
ax.axhline(1/18, color='gray', linestyle='--', alpha=0.5, label='Random chance (5.6%)')
for bar in bars1:
    height = bar.get_height()
    ax.annotate(f'{height:.2f}', xy=(bar.get_x() + bar.get_width()/2, height),
                xytext=(0, 3), textcoords='offset points', ha='center', va='bottom', fontsize=7)
for bar in bars2:
    height = bar.get_height()
    ax.annotate(f'{height:.2f}', xy=(bar.get_x() + bar.get_width()/2, height),
                xytext=(0, 3), textcoords='offset points', ha='center', va='bottom', fontsize=7)
fig.tight_layout()
fig_path = output_root / 'fig1_model_comparison_bars.png'
fig.savefig(fig_path, dpi=200)
plt.close(fig)
print(f'Figure 1 saved: {fig_path}')

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# Figure 2: Training curves (val macro F1 over epochs)
# ═══════════════════════════════════════════════════════════════════════════════
fig, axes = plt.subplots(3, 3, figsize=(14, 10), sharex=True, sharey=True)
axes = axes.flatten()

for idx, r in enumerate(results):
    metrics = json.loads(Path(r['metrics_path']).read_text())
    history = metrics['history']
    epochs = [h['epoch'] for h in history]
    train_loss = [h['train_loss'] for h in history]
    val_f1 = [h['val_macro_f1'] for h in history]
    ax = axes[idx]
    ax.plot(epochs, val_f1, 'o-', color='steelblue', label='Val F1', markersize=3)
    ax_twin = ax.twinx()
    ax_twin.plot(epochs, train_loss, 's--', color='coral', label='Train loss', markersize=3, alpha=0.7)
    ax.set_title(f"{r['model_type'].upper()} | {r['channel_mode']}", fontsize=9)
    ax.set_ylim(0, 1.0)
    ax.set_xlabel('Epoch', fontsize=8)
    ax.set_ylabel('Val F1', fontsize=8, color='steelblue')
    ax_twin.set_ylabel('Train Loss', fontsize=8, color='coral')
    ax.axhline(r['test_accuracy'], color='green', linestyle=':', alpha=0.4, linewidth=1)
    best_epoch = r['best_epoch']
    ax.axvline(best_epoch, color='gray', linestyle='--', alpha=0.4, linewidth=1)

fig.suptitle('Training Curves: Val Macro F1 (blue) + Train Loss (coral) + Best Epoch (dashed)', fontsize=11, y=1.01)
fig.tight_layout()
fig_path = output_root / 'fig2_training_curves_grid.png'
fig.savefig(fig_path, dpi=200)
plt.close(fig)
print(f'Figure 2 saved: {fig_path}')

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# Figure 3: Confusion matrices grid (3x3)
# ═══════════════════════════════════════════════════════════════════════════════
fig, axes = plt.subplots(3, 3, figsize=(14, 12))
axes = axes.flatten()

for idx, r in enumerate(results):
    metrics = json.loads(Path(r['metrics_path']).read_text())
    cm = np.array(metrics['test']['confusion_matrix'])
    ax = axes[idx]
    im = ax.imshow(cm, cmap='Blues', aspect='auto')
    ax.set_title(f"{r['model_type'].upper()} | {r['channel_mode']}", fontsize=10)
    ax.set_xlabel('Predicted', fontsize=8)
    ax.set_ylabel('True', fontsize=8)
    ticks = np.arange(num_classes)
    ax.set_xticks(ticks)
    ax.set_xticklabels([index_to_label[i] for i in ticks], fontsize=6, rotation=45)
    ax.set_yticks(ticks)
    ax.set_yticklabels([index_to_label[i] for i in ticks], fontsize=6)
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

fig.suptitle('Confusion Matrices — Full Model Sweep', fontsize=12, y=1.01)
fig.tight_layout()
fig_path = output_root / 'fig3_confusion_matrices_grid.png'
fig.savefig(fig_path, dpi=200)
plt.close(fig)
print(f'Figure 3 saved: {fig_path}')

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# Figure 4: Per-class F1 comparison (grouped bar for each class)
# ═══════════════════════════════════════════════════════════════════════════════
all_per_class = {}
for r in results:
    metrics = json.loads(Path(r['metrics_path']).read_text())
    all_per_class[f"{r['model_type']}_{r['channel_mode']}"] = metrics['test']['per_class_f1']

labels_sorted = sorted(index_to_label.values())
n_models = len(results)
x = np.arange(len(labels_sorted))
width = 0.9 / n_models

fig, ax = plt.subplots(figsize=(16, 6))
colors = plt.cm.tab20(np.linspace(0, 1, n_models))

for idx, (key, per_class) in enumerate(all_per_class.items()):
    vals = [per_class[l] for l in labels_sorted]
    offset = (idx - n_models/2 + 0.5) * width
    ax.bar(x + offset, vals, width, label=key.replace('_', ' ').upper(), color=colors[idx])

ax.set_xlabel('Activity Class')
ax.set_ylabel('Test F1')
ax.set_title('Per-Class F1 Comparison — All Model / Channel Combinations')
ax.set_xticks(x)
ax.set_xticklabels(labels_sorted)
ax.set_ylim(0, 1.0)
ax.legend(ncol=3, fontsize=7, loc='upper right')
fig.tight_layout()
fig_path = output_root / 'fig4_per_class_f1_comparison.png'
fig.savefig(fig_path, dpi=200)
plt.close(fig)
print(f'Figure 4 saved: {fig_path}')

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# Figure 5: Per-subject accuracy heatmap
# ═══════════════════════════════════════════════════════════════════════════════
subj_data = {}
subj_ids = set()
for r in results:
    metrics = json.loads(Path(r['metrics_path']).read_text())
    psa = metrics['test']['per_subject_accuracy']
    key = f"{r['model_type']}_{r['channel_mode']}"
    subj_data[key] = psa
    subj_ids.update(int(k) for k in psa.keys())

subj_ids = sorted(subj_ids)
model_keys = list(subj_data.keys())
heatmap = np.zeros((len(model_keys), len(subj_ids)))
for i, key in enumerate(model_keys):
    for j, sid in enumerate(subj_ids):
        heatmap[i, j] = subj_data[key].get(str(sid), np.nan)

fig, ax = plt.subplots(figsize=(14, 5))
im = ax.imshow(heatmap, cmap='RdYlGn', aspect='auto', vmin=0, vmax=1)
ax.set_xticks(np.arange(len(subj_ids)))
ax.set_xticklabels([str(s) for s in subj_ids], rotation=45, fontsize=8)
ax.set_yticks(np.arange(len(model_keys)))
ax.set_yticklabels([k.replace('_', ' ').upper() for k in model_keys], fontsize=8)
ax.set_xlabel('Subject ID')
ax.set_ylabel('Model / Channel')
ax.set_title('Per-Subject Test Accuracy Heatmap')
for i in range(len(model_keys)):
    for j in range(len(subj_ids)):
        val = heatmap[i, j]
        if not np.isnan(val):
            color = 'white' if val < 0.5 else 'black'
            ax.text(j, i, f'{val:.2f}', ha='center', va='center', color=color, fontsize=6)
fig.colorbar(im, ax=ax, label='Accuracy')
fig.tight_layout()
fig_path = output_root / 'fig5_per_subject_accuracy_heatmap.png'
fig.savefig(fig_path, dpi=200)
plt.close(fig)
print(f'Figure 5 saved: {fig_path}')

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# Figure 6: Channel-wise accuracy breakdown (line plot across models)
# ═══════════════════════════════════════════════════════════════════════════════
fig, ax = plt.subplots(figsize=(8, 5))
for ch in CHANNEL_MODES:
    subset = [r for r in results if r['channel_mode'] == ch]
    models = [r['model_type'].upper() for r in subset]
    accs = [r['test_accuracy'] for r in subset]
    ax.plot(models, accs, 'o-', label=ch.capitalize(), markersize=8, linewidth=2)
ax.set_ylabel('Test Accuracy')
ax.set_title('Test Accuracy by Model Architecture and Sensor Channel')
ax.legend(title='Channel')
ax.set_ylim(0, 1.0)
ax.grid(True, alpha=0.3)
for ch in CHANNEL_MODES:
    subset = [r for r in results if r['channel_mode'] == ch]
    for r in subset:
        ax.annotate(f"{r['test_accuracy']:.2f}",
                    xy=(r['model_type'].upper(), r['test_accuracy']),
                    xytext=(0, 8), textcoords='offset points',
                    ha='center', fontsize=7)
fig.tight_layout()
fig_path = output_root / 'fig6_accuracy_by_channel.png'
fig.savefig(fig_path, dpi=200)
plt.close(fig)
print(f'Figure 6 saved: {fig_path}')